# RailGuard — Stage 2: Exploratory Data Analysis

**Dataset:** MetroPT-3 — Air Production Unit (APU) compressor, Porto Metro  
**Resolution:** 1-minute resampled (from ~10s raw)  
**Period:** 2020-02-01 → 2020-09-01  

This notebook runs the Stage 2 analysis pipeline interactively and displays results inline.
All heavy computation lives in `src/` modules — this notebook is an exploration layer, not production code.

---

## Contents
1. [Environment setup](#1-environment-setup)
2. [Data quality check](#2-data-quality-check)
3. [Run preprocessing](#3-run-preprocessing)
4. [Sensor distributions](#4-sensor-distributions)
5. [Sensor time series](#5-sensor-time-series)
6. [Operational state analysis](#6-operational-state-analysis)
7. [Correlation analysis](#7-correlation-analysis)
8. [Temporal patterns](#8-temporal-patterns)
9. [Normal vs failure comparison](#9-normal-vs-failure-comparison)
10. [Failure event deep-dives](#10-failure-event-deep-dives)
11. [Key findings for Stage 3](#11-key-findings-for-stage-3)

## 1. Environment setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path (allows 'from src.xxx import ...')
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

from src.config import (
    ANALOGUE_SENSORS, DIGITAL_SENSORS, FAILURE_EVENTS, PROCESSED_DIR, RAW_CSV
)
print('Environment ready. Project root:', PROJECT_ROOT)

## 2. Data quality check

In [ ]:
from src.data.loader import load_raw
from src.data.quality import run_quality_check

# Load the raw data first
print('Loading raw data (this may take ~2 min for the full 1.5M rows)...')
df_raw = load_raw(RAW_CSV)
print(f'Loaded: {len(df_raw):,} rows, {len(df_raw.columns)} columns')
print(f'Index range: {df_raw.index.min()} → {df_raw.index.max()}')

In [ ]:
# Run quality check (uses already-loaded df, saves report to data/processed/)
quality_report = run_quality_check(df_raw, save_outputs=True)

In [ ]:
# Inspect the gap distribution
import json

gap_report = quality_report['gaps']
print(f"Total gaps > {gap_report['gap_threshold_s']}s: {gap_report['total_gaps_above_threshold']}")
print(f"Total missing time: {gap_report['total_missing_time_hours']:.1f} hours")
print(f"Dominant gap: {gap_report['dominant_gap_s']}s")
print()
print('Gap distribution (top 10 gap sizes):')
for k, v in sorted(gap_report['gap_distribution_top10'].items(), key=lambda x: -x[1]):
    print(f'  {k:>6}s × {v:>8,}')

In [ ]:
# Load and display gap table
from src.data.loader import load_parquet

gap_df = load_parquet(PROCESSED_DIR / 'timestamp_gaps.parquet')
print(f'Gaps > threshold: {len(gap_df)}')
print(f'Largest gaps:')
gap_df.nlargest(10, 'gap_hours')[['gap_start_ts', 'gap_end_ts', 'gap_hours']].to_string(index=False)

## 3. Run preprocessing

In [ ]:
from src.data.preprocessor import run_preprocessing

# Check if processed file already exists
proc_path = PROCESSED_DIR / 'processed_1min.parquet'
if proc_path.exists():
    print(f'Processed file exists at {proc_path}, loading...')
    df = load_parquet(proc_path)
else:
    print('Running preprocessing pipeline (may take 2-3 min)...')
    df = run_preprocessing(save_output=True)

print(f'Processed data: {len(df):,} rows × {len(df.columns)} columns')
print(f'Columns: {df.columns.tolist()}')
print(f'Gap rows: {df["is_gap"].sum():,} ({100*df["is_gap"].mean():.1f}% of 1-min slots)')

In [ ]:
# Preview processed data
non_gap = df[~df['is_gap']]
print(f'Non-gap rows: {len(non_gap):,}')
print()
non_gap[ANALOGUE_SENSORS].describe().round(3)

In [ ]:
# Failure window coverage
print('Failure window row counts:')
print(df['failure_id'].value_counts().to_string())
print()
failure_rows = df[df['failure_id'] != '']
total_non_gap = len(non_gap)
print(f'Failure rows (non-gap): {len(failure_rows):,}  ({100*len(failure_rows)/total_non_gap:.2f}% of total)')

In [ ]:
# Motor state distribution
print('Motor state distribution:')
print(non_gap['motor_state'].value_counts().to_string())

## 4. Sensor distributions

In [ ]:
from src.analysis.eda import plot_sensor_distributions

# This generates and saves the figure; also returns the path
path = plot_sensor_distributions(df)

# Display inline
from IPython.display import Image
Image(str(path))

## 5. Sensor time series

In [ ]:
from src.analysis.eda import plot_sensor_timeseries

path = plot_sensor_timeseries(df)
Image(str(path))

## 6. Operational state analysis

In [ ]:
from src.analysis.eda import plot_operational_states

path = plot_operational_states(df)
Image(str(path))

## 7. Correlation analysis

In [ ]:
from src.analysis.eda import plot_correlation_heatmap

path = plot_correlation_heatmap(df)
Image(str(path))

In [ ]:
# Print correlation values for discussion
corr = non_gap[ANALOGUE_SENSORS].corr().round(3)
print('Pearson correlation matrix:')
print(corr.to_string())

## 8. Temporal patterns

In [ ]:
from src.analysis.eda import plot_temporal_patterns

path = plot_temporal_patterns(df)
Image(str(path))

## 9. Normal vs failure comparison

In [ ]:
from src.analysis.eda import plot_normal_vs_failure_boxplots, plot_tp3_reservoirs

path = plot_normal_vs_failure_boxplots(df)
Image(str(path))

In [ ]:
path = plot_tp3_reservoirs(df)
Image(str(path))

In [ ]:
# Quantify normal vs failure differences
failure_mask = df['failure_id'] != ''
normal_mask  = (df['failure_id'] == '') & (~df['is_gap'])

print('Mean sensor values — Normal vs Failure:')
print(f'{"Sensor":<22} {"Normal mean":>14} {"Failure mean":>14} {"Delta":>10}')
print('-' * 62)
for col in ['H1', 'TP2', 'DV_pressure', 'Oil_temperature', 'Motor_current', 'TP3', 'Reservoirs']:
    nm = df.loc[normal_mask,  col].mean()
    fm = df.loc[failure_mask, col].mean()
    print(f'{col:<22} {nm:>14.3f} {fm:>14.3f} {fm-nm:>+10.3f}')

## 10. Failure event deep-dives

In [ ]:
from src.analysis.failure_analysis import run_failure_analysis

all_stats = run_failure_analysis(df)

In [ ]:
# Display individual event plots
from src.config import PROCESSED_DIR
FIG_DIR = PROCESSED_DIR / 'figures' / 'failure_analysis'

for fid in ['f1', 'f2', 'f3', 'f4']:
    path = FIG_DIR / f'failure_event_{fid}.png'
    if path.exists():
        print(f'--- Event {fid.upper()} ---')
        display(Image(str(path)))

In [ ]:
# Cross-event comparison
path = FIG_DIR / 'cross_event_comparison.png'
Image(str(path))

In [ ]:
# Load and display summary CSV
summary_df = pd.read_csv(PROCESSED_DIR / 'failure_event_summary.csv')
print(summary_df[['failure_id', 'period', 'n_rows', 'H1_mean', 'TP2_mean',
                   'Oil_temperature_mean', 'Motor_current_mean',
                   'DV_eletric_pct_active', 'LPS_pct_active']].to_string(index=False))

## 11. Key findings for Stage 3

The following findings from this analysis should directly inform Stage 3 (feature engineering, anomaly detection, and predictive modeling).

> **Note:** All values below are computed above in this notebook from the actual data. The cells above must be run first.


In [ ]:
# Summarise the most important findings for Stage 3
print('KEY FINDINGS FOR STAGE 3')
print('=' * 60)

# 1. Class imbalance
n_failure = int((df['failure_id'] != '').sum())
n_normal  = int((df['failure_id'] == '').sum())
print(f'\n1. CLASS BALANCE')
print(f'   Failure rows: {n_failure:,}  Normal rows: {n_normal:,}')
print(f'   Failure fraction: {100*n_failure/(n_failure+n_normal):.2f}%')

# 2. Best discriminating sensors (largest absolute mean delta)
print(f'\n2. SENSOR DISCRIMINATION (failure mean vs normal mean)')
for col in ANALOGUE_SENSORS:
    nm = df.loc[normal_mask, col].mean()
    fm = df.loc[failure_mask, col].mean()
    print(f'   {col:<22}  delta = {fm-nm:+.3f}')

# 3. LPS activity in failure vs normal
lps_normal  = 100 * df.loc[normal_mask,  'LPS'].mean()
lps_failure = 100 * df.loc[failure_mask, 'LPS'].mean()
print(f'\n3. LPS ACTIVITY')
print(f'   Normal periods:  {lps_normal:.2f}%')
print(f'   Failure periods: {lps_failure:.2f}%')

# 4. Gap summary
print(f'\n4. GAP SEGMENTS')
n_segments = int(df['gap_segment_id'].max())
print(f'   Continuous operational segments: {n_segments}')
print(f'   Total gap rows (1-min slots): {int(df["is_gap"].sum()):,}')

print()
print('See docs/stage2_findings.md for the full written findings.')